# Evaluación integral del asistente — IDA

Preparación del caso experimental de actividad universitaria. Se reproduce la ingesta y transformación del asistente sobre un RSS sintético, con una base de datos aislada. La descarga HTTP se sustituye por la lectura local del XML. En esta etapa se valida la preparación de los datos; aún no se calcula el IDA.

### 1. Dependencias

Bibliotecas utilizadas durante la preparación y validación de los datos.

In [3]:
import hashlib
import os
import re
import unicodedata
import xml.etree.ElementTree as ET
from datetime import datetime
from email.utils import parsedate_to_datetime
from pathlib import Path

import feedparser
import pandas as pd
import psycopg
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from IPython.display import display

### 2. Configuración experimental

Parámetros del caso de estudio, definidos antes de ejecutar el procesamiento. Las palabras clave corresponden al dominio temático y no se ajustan a partir del Ground Truth.

In [4]:
CASE_ID = "caso_01"
EXPECTED_DATABASE = f"ida_{CASE_ID}"
ENV_PATH = Path("..") / f".env.ida_{CASE_ID}"
FEED_PATH = Path(CASE_ID) / "feed.xml"
EXPECTED_NEWS_COUNT = 50
TOPIC = "Actividad universitaria"
SOURCE_NAME = "RSS Sintético - Actividad universitaria"
SOURCE_CATEGORY = "Educación"

KEYWORDS = [
    ("universidad", "Instituciones universitarias"),
    ("universitario", "Actividad universitaria"),
    ("facultad", "Unidades académicas"),
    ("estudiante", "Comunidad estudiantil"),
    ("docente", "Comunidad docente"),
    ("beca", "Apoyo económico estudiantil"),
    ("académico", "Actividades académicas"),
    ("educación superior", "Nivel educativo"),
    ("investigación", "Actividad científica"),
    ("carrera", "Oferta académica"),
]

feed_path = FEED_PATH.resolve(strict=True)

### 3. Conexión a la base experimental

Se verifica el entorno de trabajo antes de acceder a los datos, preservando la base original del asistente.

In [5]:
if not ENV_PATH.is_file():
    raise FileNotFoundError(f"Archivo de entorno no encontrado: {ENV_PATH}")

load_dotenv(ENV_PATH, override=True)


def get_connection() -> psycopg.Connection:
    """Abre una conexión a la base experimental configurada."""
    if os.getenv("DB_NAME") != EXPECTED_DATABASE:
        raise RuntimeError("La base configurada no corresponde al experimento.")

    connection = psycopg.connect(
        host=os.getenv("DB_HOST"),
        port=os.getenv("DB_PORT"),
        dbname=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
    )

    try:
        with connection.cursor() as cursor:
            cursor.execute("SELECT current_database();")
            database = cursor.fetchone()[0]
        if database != EXPECTED_DATABASE:
            raise RuntimeError(f"Base de datos incorrecta: {database}")
    except Exception:
        connection.close()
        raise

    return connection


with get_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute("SELECT current_database();")
        database = cursor.fetchone()[0]

print(f"Base experimental activa: {database}")

Base experimental activa: ida_caso_01


### 4. Inspección del esquema (opcional)

Consulta de solo lectura para documentar las columnas y restricciones de las tablas utilizadas. No es necesaria para ejecutar el experimento.

In [ ]:
INSPECT_SCHEMA = False

# Consulta de la estructura de la tabla de palabras clave.
schema_query = """
    SELECT
        column_name,
        data_type,
        is_nullable,
        column_default
    FROM information_schema.columns
    WHERE table_schema = %s
        AND table_name = %s
    ORDER BY ordinal_position;
"""

# Consulta de las restricciones de integridad.
constraints_query = """
    SELECT
        conname AS constraint_name,
        pg_get_constraintdef(oid) AS constraint_definition
    FROM pg_constraint
    WHERE conrelid = %s::regclass
    ORDER BY conname;
"""

if INSPECT_SCHEMA:
    # Inspeccionar las tablas involucradas en la transformación.
    silver_tables = ("rss_sources", "rss_raw", "keywords", "rss_news",
                     "rss_news_keywords")

    with get_connection() as connection:
        with connection.cursor() as cursor:
            for table_name in silver_tables:
                print(f"Estructura de {table_name}:")

                cursor.execute(
                    schema_query,
                    ("public", table_name),
                )

                display(
                    pd.DataFrame(
                        cursor.fetchall(),
                        columns=[
                            column.name
                            for column in cursor.description
                        ],
                    )
                )

                print(f"Restricciones de {table_name}:")

                cursor.execute(
                    constraints_query,
                    (f"public.{table_name}",),
                )

                display(
                    pd.DataFrame(
                        cursor.fetchall(),
                        columns=[
                            column.name
                            for column in cursor.description
                        ],
                    )
                )

## 5. Ingesta RSS — Bronze

Adaptación experimental del notebook 02: registro de la fuente y almacenamiento del XML local.

### 5.1. Registro de la fuente

Se registra el medio sintético mediante la URI del XML.

In [ ]:
def register_rss_source(
    name: str,
    feed_url: str,
    category: str,
) -> int:
    """Registra una fuente RSS y devuelve su identificador.

    Args:
        name: Nombre descriptivo de la fuente.
        feed_url: URI única del archivo RSS.
        category: Categoría temática de la fuente.

    Returns:
        Identificador de la fuente RSS registrada.

    Raises:
        RuntimeError: Si la conexión no corresponde al caso experimental.
    """
    query = """
        INSERT INTO rss_sources (
            name,
            feed_url,
            category
        )
        VALUES (%s, %s, %s)
        ON CONFLICT (feed_url) DO UPDATE
        SET
            name = EXCLUDED.name,
            category = EXCLUDED.category,
            updated_at = CURRENT_TIMESTAMP
        RETURNING rss_source_id;
    """

    with get_connection() as connection:
        with connection.cursor() as cursor:
            # Verificar la base antes de realizar escrituras.
            cursor.execute("SELECT current_database();")
            database = cursor.fetchone()[0]

            if database != EXPECTED_DATABASE:
                raise RuntimeError(
                    f"Base de datos incorrecta: {database}"
                )

            cursor.execute(query, (name, feed_url, category))
            source_id = cursor.fetchone()[0]

    return source_id


rss_source_id = register_rss_source(
    name=SOURCE_NAME,
    feed_url=feed_path.as_uri(),
    category=SOURCE_CATEGORY,
)

print(f"Fuente RSS registrada. ID: {rss_source_id}")

### 5.2. Validación de la fuente

Se comprueba la unicidad de la fuente y la conservación de su identificador.

In [ ]:
# Repetir el registro para comprobar la idempotencia.
repeated_source_id = register_rss_source(
    name=SOURCE_NAME,
    feed_url=feed_path.as_uri(),
    category=SOURCE_CATEGORY,
)

# Verificar que se conserva el identificador original.
assert repeated_source_id == rss_source_id, (
    "El identificador de la fuente ha cambiado."
)

# Consultar la cantidad de registros asociados a la fuente.
query = """
    SELECT COUNT(*)
    FROM rss_sources
    WHERE feed_url = %s;
"""

with get_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(query, (feed_path.as_uri(),))
        source_count = cursor.fetchone()[0]

# Validar que no existen registros duplicados.
assert source_count == 1, "Se detectaron fuentes duplicadas."

print(f"Identificador de la fuente: {rss_source_id}")
print(f"Cantidad de registros: {source_count}")
print("Validación de idempotencia satisfactoria.")

### 5.3. Lectura del RSS

Se lee el XML y se calcula su hash SHA-256.

In [ ]:
def read_local_rss_feed(feed_path: Path) -> dict:
    """Lee un archivo RSS local y calcula su hash SHA-256.

    Args:
        feed_path: Ruta del archivo XML de la fuente RSS.

    Returns:
        Diccionario con el contenido binario, tamaño y hash.

    Raises:
        FileNotFoundError: Si el archivo no existe.
        ValueError: Si el archivo está vacío.
    """
    content = feed_path.read_bytes()

    if not content:
        raise ValueError("El archivo RSS está vacío.")

    return {
        "content": content,
        "content_size": len(content),
        "content_hash": hashlib.sha256(content).hexdigest(),
    }


# Lectura del archivo RSS correspondiente al caso experimental.
rss_feed = read_local_rss_feed(feed_path)

# Presentación de los metadatos del archivo.
print(f"Archivo: {feed_path.name}")
print(f"Tamaño: {rss_feed['content_size']} bytes")
print(f"Hash SHA-256: {rss_feed['content_hash']}")

### 5.4. Persistencia en Bronze

Se almacena el XML original en `rss_raw`, pendiente de transformación.

In [ ]:
def insert_rss_raw(
    source_id: int,
    content: bytes,
    content_hash: str,
) -> int:
    """Almacena el contenido original de una fuente RSS.

    Args:
        source_id: Identificador de la fuente RSS.
        content: Contenido binario del archivo XML.
        content_hash: Hash SHA-256 del contenido.

    Returns:
        Cantidad de nuevos registros insertados.

    Raises:
        RuntimeError: Si la conexión no corresponde al caso experimental.
    """
    insert_query = """
        INSERT INTO rss_raw (
            fk_rss_source_id,
            raw_content,
            content_hash,
            retrieved_at,
            is_processed
        )
        VALUES (%s, %s, %s, CURRENT_TIMESTAMP, FALSE)
        ON CONFLICT (fk_rss_source_id, content_hash)
        DO NOTHING;
    """

    # Decodificar el contenido para su almacenamiento como texto.
    raw_content = content.decode("utf-8", errors="replace")

    with get_connection() as connection:
        with connection.cursor() as cursor:
            # Verificar la base de datos antes de realizar escrituras.
            cursor.execute("SELECT current_database();")
            database = cursor.fetchone()[0]

            if database != EXPECTED_DATABASE:
                raise RuntimeError(
                    f"Base de datos incorrecta: {database}"
                )

            # Persistir el contenido evitando registros duplicados.
            cursor.execute(
                insert_query,
                (source_id, raw_content, content_hash),
            )

            inserted_count = cursor.rowcount

    return inserted_count


# Persistencia del RSS sintético en la capa Bronze.
inserted_raw = insert_rss_raw(
    source_id=rss_source_id,
    content=rss_feed["content"],
    content_hash=rss_feed["content_hash"],
)

print(f"Registros RAW insertados: {inserted_raw}")

### 5.5. Validación de la ingesta

Se verifica la integridad del XML, la cantidad de noticias y el control de duplicados.

In [ ]:
# Recuperar el contenido almacenado en la capa Bronze.
query = """
    SELECT
        rss_raw_id,
        raw_content,
        content_hash,
        is_processed
    FROM rss_raw
    WHERE fk_rss_source_id = %s
        AND content_hash = %s;
"""

with get_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            query,
            (rss_source_id, rss_feed["content_hash"]),
        )
        rows = cursor.fetchall()

# Validar la existencia de un único registro RAW.
assert len(rows) == 1, "Cantidad incorrecta de registros RAW."

rss_raw_id, raw_content, stored_hash, is_processed = rows[0]

# Verificar la integridad del contenido almacenado.
calculated_hash = hashlib.sha256(
    raw_content.encode("utf-8")
).hexdigest()

assert calculated_hash == stored_hash == rss_feed["content_hash"], (
    "El contenido almacenado no coincide con el original."
)

# Verificar la cantidad de noticias contenidas en el XML.
root = ET.fromstring(raw_content)
news_count = len(root.findall("./channel/item"))

assert news_count == EXPECTED_NEWS_COUNT, "Cantidad incorrecta de noticias."
assert not is_processed, "El XML ya fue procesado."

# Repetir la ingesta para comprobar la idempotencia.
repeated_insert = insert_rss_raw(
    source_id=rss_source_id,
    content=rss_feed["content"],
    content_hash=rss_feed["content_hash"],
)

assert repeated_insert == 0, "Se generó un registro duplicado."

# Presentar los resultados de la validación.
print(f"Identificador RAW: {rss_raw_id}")
print(f"Noticias contenidas: {news_count}")
print("Integridad del contenido: OK")
print("Idempotencia de la ingesta: OK")
print("Validación de ingesta satisfactoria.")

## 6. Transformación — Silver

Adaptación experimental del notebook 03: configuración del filtro léxico, normalización y persistencia de noticias.

### 6.1. Registro de palabras clave

Se incorporan las palabras clave definidas en la configuración experimental.

In [ ]:
def register_keywords(keywords: list[tuple[str, str]]) -> int:
    """Registra las palabras clave del caso experimental.

    Args:
        keywords: Lista de palabras clave y sus descripciones.

    Returns:
        Cantidad de nuevos registros insertados.

    Raises:
        RuntimeError: Si la conexión no corresponde al caso experimental.
    """
    query = """
        INSERT INTO keywords (
            keyword,
            description
        )
        VALUES (%s, %s)
        ON CONFLICT (keyword) DO NOTHING;
    """

    with get_connection() as connection:
        with connection.cursor() as cursor:
            # Verificar la base antes de realizar escrituras.
            cursor.execute("SELECT current_database();")
            database = cursor.fetchone()[0]

            if database != EXPECTED_DATABASE:
                raise RuntimeError(
                    f"Base de datos incorrecta: {database}"
                )

            # Registrar las palabras clave evitando duplicados.
            cursor.executemany(query, keywords)
            inserted_count = cursor.rowcount

    return inserted_count


# Persistencia de las palabras clave.
inserted_keywords = register_keywords(KEYWORDS)

print(f"Palabras clave insertadas: {inserted_keywords}")

### 6.2. Validación de palabras clave

Se verifica que el conjunto registrado coincida con el previsto y se encuentre activo.

In [ ]:
# Verificar la idempotencia del registro.
repeated_insert = register_keywords(KEYWORDS)

assert repeated_insert == 0, "Se insertaron palabras duplicadas."

# Consultar las palabras clave registradas.
query = """
    SELECT
        keyword_id,
        keyword,
        description,
        is_active
    FROM keywords
    ORDER BY keyword_id;
"""

with get_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(query)

        df_keywords = pd.DataFrame(
            cursor.fetchall(),
            columns=[column.name for column in cursor.description],
        )

# Validar la cantidad, el contenido y el estado de los registros.
expected_keywords = {keyword for keyword, _ in KEYWORDS}
registered_keywords = set(df_keywords["keyword"])

assert len(df_keywords) == len(KEYWORDS), "Cantidad incorrecta de keywords."

assert registered_keywords == expected_keywords, (
    "Las palabras registradas no coinciden con las esperadas."
)

assert df_keywords["is_active"].all(), (
    "Existen palabras clave inactivas."
)

# Presentar los resultados de la validación.
display(df_keywords)

print(f"Palabras clave registradas: {len(df_keywords)}")
print("Integridad de los registros: OK")
print("Idempotencia: OK")
print("Validación satisfactoria.")

### 6.3. Recuperación del RSS

Se recupera el XML pendiente de procesamiento desde Bronze.

In [ ]:
def get_pending_rss_raw() -> list[tuple[int, int, str]]:
    """Obtiene los contenidos RSS pendientes de procesamiento.

    Returns:
        Lista de registros con el identificador RAW,
        el identificador de la fuente y el contenido XML.
    """
    select_query = """
        SELECT
            rss_raw_id,
            fk_rss_source_id,
            raw_content
        FROM rss_raw
        WHERE is_processed = FALSE
        ORDER BY rss_raw_id;
    """

    with get_connection() as connection:
        with connection.cursor() as cursor:
            cursor.execute(select_query)
            return cursor.fetchall()


# Recuperar los registros pendientes de transformación.
pending_raw = get_pending_rss_raw()

print(f"Registros RAW pendientes: {len(pending_raw)}")

### 6.4. Normalización de noticias

Se limpian los textos, se recupera el contenido y se normalizan las fechas, conservando la trazabilidad.

In [ ]:
def clean_html(value: str | None) -> str | None:
    """Elimina etiquetas HTML y normaliza los espacios.

    Args:
        value: Texto que puede contener etiquetas HTML.

    Returns:
        Texto normalizado o None si no existe contenido.
    """
    if not value:
        return None

    soup = BeautifulSoup(value, "html.parser")
    text = soup.get_text(separator=" ", strip=True)

    return " ".join(text.split())


def parse_published_date(value: str | None) -> datetime | None:
    """Convierte una fecha RSS en un objeto datetime.

    Args:
        value: Fecha de publicación de la noticia.

    Returns:
        Fecha normalizada o None si no está disponible
        o presenta un formato inválido.
    """
    if not value:
        return None

    try:
        return parsedate_to_datetime(value)
    except (TypeError, ValueError):
        return None


def normalize_rss_entry(
    entry: feedparser.FeedParserDict,
    rss_raw_id: int,
    rss_source_id: int,
) -> dict:
    """Normaliza una entrada RSS a la estructura de rss_news.

    Args:
        entry: Entrada RSS original.
        rss_raw_id: Identificador del registro RAW de origen.
        rss_source_id: Identificador de la fuente RSS.

    Returns:
        Diccionario con los atributos de la noticia normalizada.
    """
    description = clean_html(entry.get("description"))

    content_items = entry.get("content", [])
    content = None

    # Recuperar el contenido completo cuando esté disponible.
    if content_items:
        content = clean_html(content_items[0].get("value"))

    # Utilizar la descripción como contenido alternativo.
    if not content:
        content = description

    return {
        "rss_raw_id": rss_raw_id,
        "rss_source_id": rss_source_id,
        "title": clean_html(entry.get("title")),
        "description": description,
        "content": content,
        "url": entry.get("link"),
        "author": clean_html(entry.get("author")),
        "published_at": parse_published_date(
            entry.get("published")
        ),
    }


def normalize_rss_entries(
    raw_records: list[tuple[int, int, str]],
) -> list[dict]:
    """Extrae y normaliza las noticias de los registros RSS.

    Args:
        raw_records: Registros RAW pendientes de procesamiento.

    Returns:
        Lista de noticias individuales normalizadas.
    """
    normalized_entries = []

    for raw_id, source_id, raw_content in raw_records:
        feed = feedparser.parse(raw_content)

        for entry in feed.entries:
            normalized_entry = normalize_rss_entry(
                entry=entry,
                rss_raw_id=raw_id,
                rss_source_id=source_id,
            )

            normalized_entries.append(normalized_entry)

    return normalized_entries


# Transformar las noticias recuperadas desde Bronze.
normalized_entries = normalize_rss_entries(pending_raw)

df_normalized_entries = pd.DataFrame(normalized_entries)

print(f"Noticias normalizadas: {len(normalized_entries)}")

display(df_normalized_entries.head())

### 6.5. Validación de la transformación

Se comprueba la cantidad de noticias, la unicidad de las URL, los campos esenciales y la trazabilidad.

In [ ]:
# Identificar los campos esenciales de las noticias.
required_columns = [
    "rss_raw_id",
    "rss_source_id",
    "title",
    "content",
    "url",
    "published_at",
]

# Calcular los indicadores de calidad de los registros.
total_news = len(df_normalized_entries)
duplicate_urls = df_normalized_entries["url"].duplicated().sum()
null_values = df_normalized_entries[required_columns].isna().sum()

# Validar la cantidad y la integridad de las noticias.
assert total_news == EXPECTED_NEWS_COUNT, (
    "Cantidad incorrecta de noticias normalizadas."
)

assert duplicate_urls == 0, (
    "Se detectaron URL duplicadas."
)

assert null_values.sum() == 0, (
    "Se detectaron valores nulos en campos esenciales."
)

# Validar la trazabilidad con los registros de origen.
assert df_normalized_entries["rss_raw_id"].eq(rss_raw_id).all(), (
    "Se detectaron identificadores RAW incorrectos."
)

assert df_normalized_entries["rss_source_id"].eq(rss_source_id).all(), (
    "Se detectaron identificadores de fuente incorrectos."
)

# Presentar los resultados de la validación.
print(f"Noticias normalizadas: {total_news}")
print(f"URL duplicadas: {duplicate_urls}")
print(f"Valores nulos en campos esenciales: {null_values.sum()}")
print("Trazabilidad: OK")
print("Validación de transformación satisfactoria.")

### 6.6. Filtro léxico

Se detectan palabras completas en el título y el contenido, sin distinguir mayúsculas ni acentos. No se aplica lematización.

In [ ]:
def get_active_keywords() -> list[tuple[int, str]]:
    """Obtiene las palabras clave activas de PostgreSQL.

    Returns:
        Lista de identificadores y palabras clave activas.
    """
    query = """
        SELECT
            keyword_id,
            keyword
        FROM keywords
        WHERE is_active = TRUE
        ORDER BY keyword_id;
    """

    with get_connection() as connection:
        with connection.cursor() as cursor:
            cursor.execute(query)
            return cursor.fetchall()


def normalize_text(value: str | None) -> str:
    """Normaliza un texto eliminando diferencias de capitalización y acentos.

    Args:
        value: Texto que se desea normalizar.

    Returns:
        Texto normalizado.
    """
    if not value:
        return ""

    value = value.lower()

    return "".join(
        character
        for character in unicodedata.normalize("NFD", value)
        if unicodedata.category(character) != "Mn"
    )


def detect_keywords(
    news: dict,
    keywords: list[tuple[int, str]],
) -> list[dict]:
    """Detecta palabras clave completas en una noticia.

    Args:
        news: Noticia normalizada.
        keywords: Palabras clave activas configuradas.

    Returns:
        Lista de palabras clave detectadas.
    """
    searchable_text = " ".join(
        [
            news.get("title") or "",
            news.get("content") or "",
        ]
    )

    searchable_text = normalize_text(searchable_text)
    matches = []

    for keyword_id, keyword in keywords:
        normalized_keyword = normalize_text(keyword)
        pattern = rf"\b{re.escape(normalized_keyword)}\b"

        if re.search(pattern, searchable_text):
            matches.append(
                {
                    "keyword_id": keyword_id,
                    "keyword": keyword,
                }
            )

    return matches


# Recuperar las palabras clave activas.
active_keywords = get_active_keywords()

# Detectar las coincidencias para cada noticia.
keyword_results = []

for news_index, news in enumerate(normalized_entries):
    matches = detect_keywords(
        news=news,
        keywords=active_keywords,
    )

    for match in matches:
        keyword_results.append(
            {
                "news_index": news_index,
                "rss_source_id": news["rss_source_id"],
                "title": news["title"],
                "keyword_id": match["keyword_id"],
                "keyword": match["keyword"],
            }
        )

# Construir el DataFrame de coincidencias.
df_keyword_results = pd.DataFrame(
    keyword_results,
    columns=[
        "news_index",
        "rss_source_id",
        "title",
        "keyword_id",
        "keyword",
    ],
)

print(f"Palabras clave activas: {len(active_keywords)}")
print(f"Noticias analizadas: {len(normalized_entries)}")
print(f"Coincidencias detectadas: {len(df_keyword_results)}")
print(
    "Noticias con coincidencias: "
    f"{df_keyword_results['news_index'].nunique()}"
)

display(df_keyword_results.head(10))

### 6.7. Persistencia de noticias

Se almacenan las noticias normalizadas y se recuperan sus identificadores mediante la URL.

In [ ]:
def persist_news(
    news_records: list[dict],
    expected_database: str,
) -> dict[str, int]:
    """Persiste las noticias normalizadas en la capa Silver.

    Args:
        news_records: Lista de noticias normalizadas.
        expected_database: Base de datos experimental esperada.

    Returns:
        Diccionario que relaciona cada URL con su identificador.

    Raises:
        RuntimeError: Si la conexión apunta a una base incorrecta.
    """
    insert_query = """
        INSERT INTO rss_news (
            fk_rss_source_id,
            fk_rss_raw_id,
            title,
            description,
            content,
            url,
            author,
            published_at,
            content_hash
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT (url) DO NOTHING;
    """

    select_query = """
        SELECT rss_news_id
        FROM rss_news
        WHERE url = %s;
    """

    news_ids = {}

    with get_connection() as connection:
        with connection.cursor() as cursor:
            # Verificar la base de datos experimental.
            cursor.execute("SELECT current_database();")
            database = cursor.fetchone()[0]

            if database != expected_database:
                raise RuntimeError(
                    f"Base de datos incorrecta: {database}"
                )

            for news in news_records:
                # Calcular el hash del contenido normalizado.
                content = news.get("content") or ""
                content_hash = hashlib.sha256(
                    content.encode("utf-8")
                ).hexdigest()

                cursor.execute(
                    insert_query,
                    (
                        news["rss_source_id"],
                        news["rss_raw_id"],
                        news["title"],
                        news["description"],
                        news["content"],
                        news["url"],
                        news["author"],
                        news["published_at"],
                        content_hash,
                    ),
                )

                # Recuperar el ID de la noticia nueva o existente.
                cursor.execute(select_query, (news["url"],))
                news_ids[news["url"]] = cursor.fetchone()[0]

    return news_ids


# Persistir las noticias normalizadas.
news_ids = persist_news(
    news_records=normalized_entries,
    expected_database=EXPECTED_DATABASE,
)

print(f"Noticias persistidas o recuperadas: {len(news_ids)}")

### 6.8. Persistencia de coincidencias

Se vinculan las noticias con las palabras clave detectadas, evitando duplicados.

In [ ]:
def persist_news_keywords(
    keyword_results: list[dict],
    news_records: list[dict],
    news_ids: dict[str, int],
    expected_database: str,
) -> int:
    """Persiste las relaciones entre noticias y palabras clave.

    Args:
        keyword_results: Coincidencias detectadas por el filtro léxico.
        news_records: Noticias normalizadas.
        news_ids: Identificadores de noticias asociados a sus URL.
        expected_database: Base de datos experimental esperada.

    Returns:
        Cantidad de nuevas relaciones insertadas.

    Raises:
        RuntimeError: Si la conexión apunta a una base incorrecta.
    """
    query = """
        INSERT INTO rss_news_keywords (
            fk_rss_news_id,
            fk_keyword_id
        )
        VALUES (%s, %s)
        ON CONFLICT (fk_rss_news_id, fk_keyword_id)
        DO NOTHING;
    """

    # Construir las relaciones utilizando los identificadores persistidos.
    relations = []

    for match in keyword_results:
        news_url = news_records[match["news_index"]]["url"]

        relations.append(
            (
                news_ids[news_url],
                match["keyword_id"],
            )
        )

    with get_connection() as connection:
        with connection.cursor() as cursor:
            # Verificar la base de datos experimental.
            cursor.execute("SELECT current_database();")
            database = cursor.fetchone()[0]

            if database != expected_database:
                raise RuntimeError(
                    f"Base de datos incorrecta: {database}"
                )

            # Persistir las relaciones evitando duplicados.
            cursor.executemany(query, relations)
            inserted_count = cursor.rowcount

    return inserted_count


# Persistir las coincidencias del filtro léxico.
inserted_relations = persist_news_keywords(
    keyword_results=keyword_results,
    news_records=normalized_entries,
    news_ids=news_ids,
    expected_database=EXPECTED_DATABASE,
)

print(f"Relaciones insertadas: {inserted_relations}")

In [7]:
with get_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute("""
            SELECT
                (SELECT current_database()) AS database,
                (SELECT COUNT(*) FROM rss_raw) AS raw_count,
                (SELECT COUNT(*) FROM rss_news) AS news_count,
                (SELECT COUNT(*) FROM rss_news_keywords) AS matches,
                (
                    SELECT COUNT(DISTINCT fk_rss_news_id)
                    FROM rss_news_keywords
                ) AS matched_news;
        """)

        display(
            pd.DataFrame(
                cursor.fetchall(),
                columns=[column.name for column in cursor.description],
            )
        )

,database,raw_count,news_count,matches,matched_news
0,ida_caso_01,1,50,23,17


### Finalización de la transformación

Una vez validada la persistencia de las noticias y sus
relaciones con las palabras clave, se actualiza el estado
del registro RAW para indicar que su transformación
ha finalizado satisfactoriamente.

In [8]:
def mark_raw_as_processed(
    raw_id: int,
    expected_database: str,
) -> int:
    """Marca un registro RAW como procesado.

    Args:
        raw_id: Identificador del registro RAW.
        expected_database: Base de datos experimental esperada.

    Returns:
        Cantidad de registros actualizados.

    Raises:
        RuntimeError: Si la conexión apunta a una base incorrecta.
    """
    query = """
        UPDATE rss_raw
        SET is_processed = TRUE
        WHERE rss_raw_id = %s
            AND is_processed = FALSE;
    """

    with get_connection() as connection:
        with connection.cursor() as cursor:
            # Verificar la base antes de actualizar el registro.
            cursor.execute("SELECT current_database();")
            database = cursor.fetchone()[0]

            if database != expected_database:
                raise RuntimeError(
                    f"Base de datos incorrecta: {database}"
                )

            cursor.execute(query, (raw_id,))
            updated_count = cursor.rowcount

    return updated_count


# Finalizar el procesamiento del registro experimental.
updated_count = mark_raw_as_processed(
    raw_id=1,
    expected_database="ida_caso_01",
)

print(f"Registros RAW actualizados: {updated_count}")

Registros RAW actualizados: 1


## Evaluación del Índice de Desempeño del Asistente (IDA)

### Evaluación del monitoreo

Se utiliza el conjunto de referencia del caso experimental
para contrastar las noticias seleccionadas por el asistente
mediante el filtro léxico y la recuperación semántica NLI.

Esta comparación permite calcular los indicadores de
cobertura y pertinencia de la dimensión de monitoreo.

In [10]:
import json
def load_ground_truth(file_path: Path) -> dict | list:
    """Carga los datos de referencia del caso experimental.

    Args:
        file_path: Ruta del archivo JSON de referencia.

    Returns:
        Contenido del archivo de referencia.
    """
    with file_path.open("r", encoding="utf-8") as file:
        return json.load(file)


# Cargar los datos de referencia del monitoreo.
ground_truth_path = feed_path.parent / "ground_truth_news.json"
ground_truth_news = load_ground_truth(ground_truth_path)

# Inspeccionar la estructura de los datos de referencia.
print(f"Tipo de estructura: {type(ground_truth_news).__name__}")

if isinstance(ground_truth_news, dict):
    print(f"Claves disponibles: {list(ground_truth_news.keys())}")
else:
    print(f"Cantidad de registros: {len(ground_truth_news)}")
    display(pd.DataFrame(ground_truth_news).head())

Tipo de estructura: dict
Claves disponibles: ['topic', 'source_name', 'news']


### Estructuración de los datos de referencia

Se transforma el conjunto de noticias de referencia en
una estructura tabular para facilitar su validación y
posterior comparación con los resultados del asistente.

In [11]:
# Construir el DataFrame de referencia.
ground_truth_df = pd.DataFrame(ground_truth_news["news"])

# Validar la cantidad de noticias.
assert len(ground_truth_df) == 50, (
    "Cantidad inesperada de noticias de referencia."
)

print(f"Noticias de referencia: {len(ground_truth_df)}")
print(f"Columnas disponibles: {ground_truth_df.columns.tolist()}")

display(ground_truth_df.head())

Noticias de referencia: 50
Columnas disponibles: ['news_id', 'url', 'is_relevant', 'sentiment', 'entities', 'key_facts']


,news_id,url,is_relevant,sentiment,entities,key_facts
0,CU01-001,https://caso01.example.invalid/noticias/001,True,positive,[{'entity_text': 'Universidad del Valle Claro'...,[La ampliación a 240 becas fue celebrada por e...
1,CU01-002,https://caso01.example.invalid/noticias/002,True,positive,[{'entity_text': 'Universidad del Valle Claro'...,[La comunidad recibió con entusiasmo la extens...
2,CU01-003,https://caso01.example.invalid/noticias/003,True,positive,[{'entity_text': 'Universidad del Valle Claro'...,[La renovación de equipos beneficia a 120 estu...
3,CU01-004,https://caso01.example.invalid/noticias/004,True,negative,[{'entity_text': 'Universidad del Valle Claro'...,[La interrupción perjudicó a estudiantes y gen...
4,CU01-005,https://caso01.example.invalid/noticias/005,True,negative,[{'entity_text': 'Universidad del Valle Claro'...,[El retraso de dos meses agravó dificultades e...


### Recuperación de los resultados del monitoreo

Se recuperan los resultados de la clasificación temática
almacenados en PostgreSQL. Una noticia se considera
seleccionada cuando presenta una coincidencia léxica
o ha sido recuperada mediante el modelo NLI.

In [14]:
def fetch_monitoring_results(topic: str) -> pd.DataFrame:
    """Recupera los resultados del monitoreo experimental.

    Args:
        topic: Temática evaluada.

    Returns:
        DataFrame con los resultados de selección de noticias.
    """
    query = """
        SELECT
            r.url,
            n.keyword_match,
            n.nli_recovered,
            (n.keyword_match OR n.nli_recovered) AS is_selected
        FROM nli_news AS n
        INNER JOIN rss_news AS r
            ON n.rss_news_id = r.rss_news_id
        WHERE n.topic = %s;
    """

    with get_connection() as connection:
        with connection.cursor() as cursor:
            cursor.execute(query, (topic,))
            records = cursor.fetchall()
            columns = [column.name for column in cursor.description]

    return pd.DataFrame(records, columns=columns)


# Recuperar los resultados de la temática experimental.
monitoring_df = fetch_monitoring_results(
    topic="actividad_universitaria"
)

print(f"Noticias procesadas: {len(monitoring_df)}")
print(f"Noticias seleccionadas: {monitoring_df['is_selected'].sum()}")

display(monitoring_df.head())

Noticias procesadas: 50
Noticias seleccionadas: 21


,url,keyword_match,nli_recovered,is_selected
0,https://caso01.example.invalid/noticias/001,True,False,True
1,https://caso01.example.invalid/noticias/002,True,False,True
2,https://caso01.example.invalid/noticias/003,True,False,True
3,https://caso01.example.invalid/noticias/004,True,False,True
4,https://caso01.example.invalid/noticias/005,True,False,True


### Integración de los resultados experimentales

Se integran los datos de referencia y los resultados
del monitoreo mediante la URL como identificador común.

Se valida la correspondencia uno a uno entre ambos
conjuntos para garantizar la integridad de la evaluación.

In [15]:
# Integrar los datos de referencia y los resultados del monitoreo.
monitoring_evaluation_df = ground_truth_df[
    ["news_id", "url", "is_relevant"]
].merge(
    monitoring_df,
    on="url",
    how="outer",
    validate="one_to_one",
    indicator=True,
)

# Validar la integridad del conjunto experimental.
assert len(monitoring_evaluation_df) == 50
assert monitoring_evaluation_df["_merge"].eq("both").all()
assert monitoring_evaluation_df["is_relevant"].sum() == 30
assert monitoring_evaluation_df["is_selected"].sum() == 21

# Eliminar la columna auxiliar de validación.
monitoring_evaluation_df = monitoring_evaluation_df.drop(
    columns="_merge"
)

print(f"Noticias integradas: {len(monitoring_evaluation_df)}")
print(f"Noticias relevantes: {monitoring_evaluation_df['is_relevant'].sum()}")
print(f"Noticias seleccionadas: {monitoring_evaluation_df['is_selected'].sum()}")

display(monitoring_evaluation_df.head())

Noticias integradas: 50
Noticias relevantes: 30
Noticias seleccionadas: 21


,news_id,url,is_relevant,keyword_match,nli_recovered,is_selected
0,CU01-001,https://caso01.example.invalid/noticias/001,True,True,False,True
1,CU01-002,https://caso01.example.invalid/noticias/002,True,True,False,True
2,CU01-003,https://caso01.example.invalid/noticias/003,True,True,False,True
3,CU01-004,https://caso01.example.invalid/noticias/004,True,True,False,True
4,CU01-005,https://caso01.example.invalid/noticias/005,True,True,False,True


### Construcción de la matriz de confusión

Se contrastan las etiquetas de relevancia del conjunto
de referencia con las decisiones de selección del asistente.

La matriz de confusión permite identificar los aciertos
y errores del monitoreo integrado y proporciona la base
para calcular los indicadores de cobertura y pertinencia.

In [16]:
# Obtener las etiquetas esperadas y las decisiones del asistente.
y_true = monitoring_evaluation_df["is_relevant"]
y_pred = monitoring_evaluation_df["is_selected"]

# Calcular los componentes de la matriz de confusión.
tp = int((y_true & y_pred).sum())
fp = int((~y_true & y_pred).sum())
tn = int((~y_true & ~y_pred).sum())
fn = int((y_true & ~y_pred).sum())

# Validar la consistencia de los resultados.
assert tp + fp + tn + fn == len(monitoring_evaluation_df)

# Construir la matriz de confusión.
monitoring_confusion_matrix = pd.DataFrame(
    [[tp, fn], [fp, tn]],
    index=["Relevante", "Irrelevante"],
    columns=["Seleccionada", "No seleccionada"],
)

display(monitoring_confusion_matrix)

print(f"Verdaderos positivos (VP): {tp}")
print(f"Falsos positivos (FP): {fp}")
print(f"Verdaderos negativos (VN): {tn}")
print(f"Falsos negativos (FN): {fn}")

,Seleccionada,No seleccionada
Relevante,20,10
Irrelevante,1,19


Verdaderos positivos (VP): 20
Falsos positivos (FP): 1
Verdaderos negativos (VN): 19
Falsos negativos (FN): 10


### Cálculo de los indicadores de monitoreo

Se calculan los indicadores de cobertura (M1) y
pertinencia (M2) a partir de la matriz de confusión.

La dimensión de monitoreo (M) se obtiene mediante
la media aritmética de ambos indicadores y se contrasta
con el umbral de aceptación experimental establecido.

In [17]:
# Definir el umbral de aceptación experimental.
IDA_THRESHOLD = 0.70

# Calcular los indicadores de monitoreo.
m1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
m2 = tp / (tp + fp) if (tp + fp) > 0 else 0.0

# Calcular la dimensión de monitoreo.
m = (m1 + m2) / 2

# Evaluar el cumplimiento del criterio de aceptación.
monitoring_accepted = m >= IDA_THRESHOLD

print(f"M1 - Cobertura: {m1:.4f}")
print(f"M2 - Pertinencia: {m2:.4f}")
print(f"M  - Monitoreo: {m:.4f}")
print(f"Criterio de aceptación: {monitoring_accepted}")

M1 - Cobertura: 0.6667
M2 - Pertinencia: 0.9524
M  - Monitoreo: 0.8095
Criterio de aceptación: True


In [18]:
# Definir la ruta de persistencia de los resultados.
ida_results_path = feed_path.parent / "resultados_ida.json"

# Recuperar resultados anteriores, si existen.
ida_results = (
    load_ground_truth(ida_results_path)
    if ida_results_path.exists()
    else {}
)

# Incorporar los resultados del monitoreo.
ida_results["monitoring"] = {
    "m1_coverage": m1,
    "m2_pertinence": m2,
    "score": m,
    "threshold": IDA_THRESHOLD,
    "accepted": bool(monitoring_accepted),
    "confusion_matrix": {
        "true_positives": tp,
        "false_positives": fp,
        "true_negatives": tn,
        "false_negatives": fn,
    },
}

# Persistir los resultados experimentales.
with ida_results_path.open("w", encoding="utf-8") as file:
    json.dump(
        ida_results,
        file,
        ensure_ascii=False,
        indent=4,
    )

print(f"Resultados almacenados en: {ida_results_path}")

Resultados almacenados en: E:\Trabajo final UGR\communication-intelligence-assistant\evaluacion_ida\caso_01\resultados_ida.json


## Evaluación individual del modelo de sentimientos

### Recuperación de las clasificaciones

Se recuperan las clasificaciones de sentimiento almacenadas
en PostgreSQL, incorporando la URL de cada noticia como
identificador común para su comparación con los datos
de referencia del caso experimental.

In [19]:
def fetch_sentiment_results() -> pd.DataFrame:
    """Recupera las clasificaciones de sentimiento.

    Returns:
        DataFrame con las noticias y sus sentimientos predichos.
    """
    query = """
        SELECT
            r.url,
            s.sentiment AS predicted_sentiment,
            s.positive_score,
            s.neutral_score,
            s.negative_score
        FROM rss_sentiments AS s
        INNER JOIN rss_news AS r
            ON s.fk_rss_news_id = r.rss_news_id;
    """

    with get_connection() as connection:
        with connection.cursor() as cursor:
            cursor.execute(query)
            records = cursor.fetchall()
            columns = [column.name for column in cursor.description]

    return pd.DataFrame(records, columns=columns)


# Recuperar las clasificaciones de sentimiento.
sentiment_df = fetch_sentiment_results()

# Validar la integridad de los resultados.
assert len(sentiment_df) == 50
assert sentiment_df["url"].is_unique
assert not sentiment_df.isna().any().any()

print(f"Noticias clasificadas: {len(sentiment_df)}")

display(sentiment_df.head())

Noticias clasificadas: 50


,url,predicted_sentiment,positive_score,neutral_score,negative_score
0,https://caso01.example.invalid/noticias/001,positive,0.667676,0.310604,0.021720
1,https://caso01.example.invalid/noticias/002,positive,0.821869,0.162341,0.015790
2,https://caso01.example.invalid/noticias/003,positive,0.791466,0.190898,0.017636
3,https://caso01.example.invalid/noticias/004,negative,0.017354,0.129853,0.852793
4,https://caso01.example.invalid/noticias/005,negative,0.023194,0.200152,0.776654


### Integración de las clasificaciones con los datos de referencia

Se integran las clasificaciones de sentimiento generadas
por el modelo con las etiquetas del conjunto de referencia,
utilizando la URL como identificador común.

Se verifica la correspondencia uno a uno y la validez de
las categorías antes de calcular las métricas de evaluación.

In [20]:
# Definir las categorías válidas de sentimiento.
SENTIMENT_LABELS = ["negative", "neutral", "positive"]

# Integrar las predicciones con los datos de referencia.
sentiment_evaluation_df = ground_truth_df[
    ["news_id", "url", "sentiment"]
].merge(
    sentiment_df,
    on="url",
    how="outer",
    validate="one_to_one",
    indicator=True,
)

# Validar la integridad del conjunto experimental.
assert len(sentiment_evaluation_df) == 50
assert sentiment_evaluation_df["_merge"].eq("both").all()

assert sentiment_evaluation_df["sentiment"].isin(
    SENTIMENT_LABELS
).all()

assert sentiment_evaluation_df["predicted_sentiment"].isin(
    SENTIMENT_LABELS
).all()

# Eliminar la columna auxiliar de validación.
sentiment_evaluation_df = sentiment_evaluation_df.drop(
    columns="_merge"
)

print(f"Noticias integradas: {len(sentiment_evaluation_df)}")

display(sentiment_evaluation_df.head())

Noticias integradas: 50


,news_id,url,sentiment,predicted_sentiment,positive_score,neutral_score,negative_score
0,CU01-001,https://caso01.example.invalid/noticias/001,positive,positive,0.667676,0.310604,0.021720
1,CU01-002,https://caso01.example.invalid/noticias/002,positive,positive,0.821869,0.162341,0.015790
2,CU01-003,https://caso01.example.invalid/noticias/003,positive,positive,0.791466,0.190898,0.017636
3,CU01-004,https://caso01.example.invalid/noticias/004,negative,negative,0.017354,0.129853,0.852793
4,CU01-005,https://caso01.example.invalid/noticias/005,negative,negative,0.023194,0.200152,0.776654


### Construcción de la matriz de confusión multiclase

Se comparan las etiquetas de sentimiento de referencia
con las predicciones del modelo mediante una matriz
de confusión multiclase.

Esta permite identificar los aciertos y los errores
de clasificación entre las tres categorías de sentimiento.

In [22]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)
# Obtener las etiquetas de referencia y las predicciones.
sentiment_y_true = sentiment_evaluation_df["sentiment"]
sentiment_y_pred = sentiment_evaluation_df["predicted_sentiment"]

# Construir la matriz de confusión multiclase.
sentiment_confusion_matrix = pd.DataFrame(
    confusion_matrix(
        sentiment_y_true,
        sentiment_y_pred,
        labels=SENTIMENT_LABELS,
    ),
    index=SENTIMENT_LABELS,
    columns=SENTIMENT_LABELS,
)

# Identificar los ejes de la matriz.
sentiment_confusion_matrix.index.name = "Real"
sentiment_confusion_matrix.columns.name = "Predicción"

# Validar la cantidad de noticias evaluadas.
assert sentiment_confusion_matrix.to_numpy().sum() == 50

display(sentiment_confusion_matrix)

Predicción,negative,neutral,positive
Real,,,
negative,16,0,0
neutral,0,16,1
positive,0,0,17


### Evaluación del desempeño predictivo

Se calculan las métricas de Accuracy, Precision, Recall
y F1-score para evaluar el desempeño predictivo del
modelo de clasificación de sentimientos.

Se utiliza F1-macro como métrica principal, otorgando
la misma importancia a cada categoría de sentimiento.

In [23]:
# Calcular las métricas de clasificación.
sentiment_report = classification_report(
    sentiment_y_true,
    sentiment_y_pred,
    labels=SENTIMENT_LABELS,
    output_dict=True,
    zero_division=0,
)

sentiment_accuracy = accuracy_score(
    sentiment_y_true,
    sentiment_y_pred,
)

sentiment_f1_macro = sentiment_report["macro avg"]["f1-score"]

# Construir la tabla de métricas por categoría.
sentiment_metrics_df = pd.DataFrame(
    sentiment_report
).transpose()

display(sentiment_metrics_df.round(4))

print(f"Accuracy: {sentiment_accuracy:.4f}")
print(f"F1-macro: {sentiment_f1_macro:.4f}")

,precision,recall,f1-score,support
negative,1.0000,1.0000,1.0000,16.00
neutral,1.0000,0.9412,0.9697,17.00
positive,0.9444,1.0000,0.9714,17.00
accuracy,0.9800,0.9800,0.9800,0.98
macro avg,0.9815,0.9804,0.9804,50.00
weighted avg,0.9811,0.9800,0.9800,50.00


Accuracy: 0.9800
F1-macro: 0.9804


In [24]:
# Definir la ruta de los resultados individuales.
model_results_path = feed_path.parent / "resultados_modelos.json"

# Recuperar resultados anteriores, si existen.
model_results = (
    load_ground_truth(model_results_path)
    if model_results_path.exists()
    else {}
)

# Incorporar los resultados del modelo de sentimientos.
model_results["sentiment"] = {
    "sample_size": len(sentiment_evaluation_df),
    "accuracy": float(sentiment_accuracy),
    "precision_macro": sentiment_report["macro avg"]["precision"],
    "recall_macro": sentiment_report["macro avg"]["recall"],
    "f1_macro": float(sentiment_f1_macro),
    "classification_report": sentiment_report,
    "confusion_matrix": {
        "labels": SENTIMENT_LABELS,
        "values": sentiment_confusion_matrix.to_numpy().tolist(),
    },
}

# Persistir los resultados experimentales.
with model_results_path.open("w", encoding="utf-8") as file:
    json.dump(
        model_results,
        file,
        ensure_ascii=False,
        indent=4,
    )

print(f"Resultados almacenados en: {model_results_path}")

Resultados almacenados en: E:\Trabajo final UGR\communication-intelligence-assistant\evaluacion_ida\caso_01\resultados_modelos.json


## Evaluación individual del modelo NER

### Recuperación de las entidades extraídas

Se recuperan las entidades reconocidas por el modelo NER
y almacenadas en PostgreSQL, incorporando la URL de
la noticia como identificador para su comparación con
el conjunto de referencia experimental.

In [25]:
def fetch_ner_results() -> pd.DataFrame:
    """Recupera las entidades reconocidas por el modelo NER.

    Returns:
        DataFrame con las entidades y sus noticias de origen.
    """
    query = """
        SELECT
            r.url,
            e.entity_text,
            e.entity_type,
            e.confidence_score,
            e.model_name
        FROM rss_entities AS e
        INNER JOIN rss_news AS r
            ON e.fk_rss_news_id = r.rss_news_id;
    """

    with get_connection() as connection:
        with connection.cursor() as cursor:
            cursor.execute(query)
            records = cursor.fetchall()
            columns = [column.name for column in cursor.description]

    return pd.DataFrame(records, columns=columns)


# Recuperar las entidades del conjunto experimental.
ner_df = fetch_ner_results()

# Validar la integridad de los resultados.
assert len(ner_df) == 59
assert not ner_df.isna().any().any()
assert ner_df["confidence_score"].between(0, 1).all()
assert ner_df["model_name"].nunique() == 1

print(f"Entidades recuperadas: {len(ner_df)}")
print(f"Noticias con entidades: {ner_df['url'].nunique()}")

display(ner_df.head())

Entidades recuperadas: 59
Noticias con entidades: 40


,url,entity_text,entity_type,confidence_score,model_name
0,https://caso01.example.invalid/noticias/001,Universidad del Valle Claro,organization,0.959269,urchade/gliner_multi-v2.1
1,https://caso01.example.invalid/noticias/001,estudiantes,person,0.822705,urchade/gliner_multi-v2.1
2,https://caso01.example.invalid/noticias/002,Universidad del Valle Claro,organization,0.913808,urchade/gliner_multi-v2.1
3,https://caso01.example.invalid/noticias/003,Universidad del Valle Claro,organization,0.967639,urchade/gliner_multi-v2.1
4,https://caso01.example.invalid/noticias/003,120 estudiantes,person,0.789145,urchade/gliner_multi-v2.1


### Evaluación del desempeño predictivo del modelo NER

Se comparan las entidades extraídas por el modelo con
las entidades del conjunto de referencia, considerando
la URL, el texto y la categoría de cada entidad.

Se calculan los verdaderos positivos, falsos positivos
y falsos negativos mediante operaciones de conjuntos,
obteniendo las métricas de Precision, Recall y F1-score
a nivel global y por categoría.

In [26]:
NER_LABELS = ["person", "organization", "location", "product"]
ENTITY_COLUMNS = ["url", "entity_text", "entity_type"]


def normalize_entities(
    entities_df: pd.DataFrame,
) -> set[tuple[str, str, str]]:
    """Normaliza las entidades para su comparación.

    Args:
        entities_df: DataFrame con las entidades.

    Returns:
        Conjunto de entidades únicas por noticia.
    """
    normalized_df = entities_df[ENTITY_COLUMNS].copy()

    assert not normalized_df.isna().any().any()
    assert normalized_df["entity_type"].isin(NER_LABELS).all()

    normalized_df["entity_text"] = (
        normalized_df["entity_text"]
        .str.casefold()
        .str.split()
        .str.join(" ")
    )

    assert normalized_df["entity_text"].ne("").all()

    return set(normalized_df.itertuples(index=False, name=None))


def calculate_ner_metrics(
    expected: set,
    predicted: set,
) -> dict:
    """Calcula las métricas de extracción de entidades.

    Args:
        expected: Conjunto de entidades de referencia.
        predicted: Conjunto de entidades predichas.

    Returns:
        Diccionario con los conteos y métricas de evaluación.
    """
    tp = len(expected & predicted)
    fp = len(predicted - expected)
    fn = len(expected - predicted)

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall
        else 0.0
    )

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "support": len(expected),
    }


# Expandir las entidades del conjunto de referencia.
ground_truth_ner_df = pd.DataFrame(
    [
        {
            "url": news["url"],
            "entity_text": entity["entity_text"],
            "entity_type": entity["entity_type"],
        }
        for news in ground_truth_news["news"]
        for entity in news["entities"]
    ],
    columns=ENTITY_COLUMNS,
)

# Validar que las predicciones pertenecen al caso experimental.
assert ner_df["url"].isin(ground_truth_df["url"]).all()

# Normalizar ambos conjuntos.
expected_entities = normalize_entities(ground_truth_ner_df)
predicted_entities = normalize_entities(ner_df)

# Calcular las métricas por categoría.
ner_metrics = {
    label: calculate_ner_metrics(
        {entity for entity in expected_entities if entity[2] == label},
        {entity for entity in predicted_entities if entity[2] == label},
    )
    for label in NER_LABELS
}

ner_metrics_df = pd.DataFrame(ner_metrics).transpose()

# Calcular las métricas globales.
ner_micro_metrics = calculate_ner_metrics(
    expected_entities,
    predicted_entities,
)

ner_f1_macro = float(ner_metrics_df["f1_score"].mean())

# Presentar los resultados.
display(ner_metrics_df.round(4))

print(f"Entidades esperadas: {len(expected_entities)}")
print(f"Entidades predichas: {len(predicted_entities)}")
print(f"Precision micro: {ner_micro_metrics['precision']:.4f}")
print(f"Recall micro: {ner_micro_metrics['recall']:.4f}")
print(f"F1-micro: {ner_micro_metrics['f1_score']:.4f}")
print(f"F1-macro: {ner_f1_macro:.4f}")

,tp,fp,fn,precision,recall,f1_score,support
person,0.0,12.0,0.0,0.0000,0.0,0.0000,0.0
organization,30.0,9.0,0.0,0.7692,1.0,0.8696,30.0
location,0.0,5.0,0.0,0.0000,0.0,0.0000,0.0
product,0.0,3.0,0.0,0.0000,0.0,0.0000,0.0


Entidades esperadas: 30
Entidades predichas: 59
Precision micro: 0.5085
Recall micro: 1.0000
F1-micro: 0.6742
F1-macro: 0.2174


In [27]:
# Recuperar los resultados individuales anteriores.
model_results = load_ground_truth(model_results_path)

# Incorporar los resultados del modelo NER.
model_results["ner"] = {
    "sample_size": len(ground_truth_df),
    "expected_entities": len(expected_entities),
    "predicted_entities": len(predicted_entities),
    "micro_metrics": ner_micro_metrics,
    "f1_macro": ner_f1_macro,
    "metrics_by_category": ner_metrics,
}

# Persistir los resultados experimentales.
with model_results_path.open("w", encoding="utf-8") as file:
    json.dump(
        model_results,
        file,
        ensure_ascii=False,
        indent=4,
    )

print(f"Resultados almacenados en: {model_results_path}")

Resultados almacenados en: E:\Trabajo final UGR\communication-intelligence-assistant\evaluacion_ida\caso_01\resultados_modelos.json


### Hallazgos y limitaciones del modelo NER

El modelo de reconocimiento de entidades nombradas obtuvo una precisión micro del 50,85%, una exhaustividad (Recall) del 100% y un F1-micro del 67,42%.

De las 59 entidades extraídas, 30 coincidieron con el conjunto de referencia y 29 fueron clasificadas como falsos positivos. No se registraron falsos negativos.

Estos resultados evidencian una capacidad de recuperación completa de las entidades esperadas, aunque acompañada de una tendencia a la sobreextracción. Se observaron errores asociados a la identificación de sustantivos comunes y referencias colectivas como entidades nombradas, lo que redujo la precisión del modelo.

Por otra parte, el conjunto experimental presenta una limitación en la representatividad de las categorías, dado que contiene exclusivamente ejemplos positivos de organizaciones. Esta característica condiciona la interpretación del F1-macro, cuyo valor fue del 21,74%, e impide evaluar integralmente la capacidad de recuperación de personas, ubicaciones y productos.

Se conservaron los parámetros originales del modelo y las anotaciones del conjunto de referencia, evitando introducir ajustes posteriores a la observación de los resultados experimentales.

## Evaluación de la dimensión de análisis (A)

Se evalúa la capacidad analítica del asistente mediante
tres consultas experimentales ejecutadas a través de
Open WebUI, utilizando las herramientas del servidor MCP.

Las respuestas obtenidas se contrastan con los datos
de referencia para determinar la correctitud y
completitud de los resultados analíticos.

In [28]:
# Registrar los resultados de las consultas experimentales.
analysis_evaluation_df = pd.DataFrame(
    [
        {
            "query_id": "A01",
            "expected_units": 1,
            "provided_units": 1,
            "correct_units": 1,
            "tool": "news_count_by_source",
        },
        {
            "query_id": "A02",
            "expected_units": 3,
            "provided_units": 3,
            "correct_units": 1,
            "tool": "sentiment_distribution",
        },
        {
            "query_id": "A03",
            "expected_units": 3,
            "provided_units": 3,
            "correct_units": 3,
            "tool": "top_entities",
        },
    ]
)

display(analysis_evaluation_df)

,query_id,expected_units,provided_units,correct_units,tool
0,A01,1,1,1,news_count_by_source
1,A02,3,3,1,sentiment_distribution
2,A03,3,3,3,top_entities


### Cálculo de los indicadores de análisis

Se calculan los indicadores de correctitud analítica
(A1) y completitud analítica (A2) a partir de los
resultados de las consultas experimentales.

La dimensión de análisis (A) se obtiene mediante
la media aritmética de ambos indicadores y se contrasta
con el umbral de aceptación establecido.

In [29]:
# Consolidar las unidades de evaluación.
expected_units = int(analysis_evaluation_df["expected_units"].sum())
provided_units = int(analysis_evaluation_df["provided_units"].sum())
correct_units = int(analysis_evaluation_df["correct_units"].sum())

# Validar la consistencia de los resultados.
assert 0 <= correct_units <= min(expected_units, provided_units)

# Calcular los indicadores de análisis.
a1 = correct_units / provided_units if provided_units else 0.0
a2 = correct_units / expected_units if expected_units else 0.0

# Calcular la dimensión de análisis.
a = (a1 + a2) / 2

# Evaluar el cumplimiento del criterio de aceptación.
analysis_accepted = a >= IDA_THRESHOLD

print(f"A1 - Correctitud analítica: {a1:.4f}")
print(f"A2 - Completitud analítica: {a2:.4f}")
print(f"A  - Análisis: {a:.4f}")
print(f"Criterio de aceptación: {analysis_accepted}")

A1 - Correctitud analítica: 0.7143
A2 - Completitud analítica: 0.7143
A  - Análisis: 0.7143
Criterio de aceptación: True


In [30]:
# Recuperar los resultados anteriores del IDA.
ida_results = load_ground_truth(ida_results_path)

# Incorporar los resultados de la dimensión de análisis.
ida_results["analysis"] = {
    "a1_correctness": a1,
    "a2_completeness": a2,
    "score": a,
    "threshold": IDA_THRESHOLD,
    "accepted": bool(analysis_accepted),
    "expected_units": expected_units,
    "provided_units": provided_units,
    "correct_units": correct_units,
    "queries": analysis_evaluation_df.to_dict(orient="records"),
}

# Persistir los resultados experimentales.
with ida_results_path.open("w", encoding="utf-8") as file:
    json.dump(
        ida_results,
        file,
        ensure_ascii=False,
        indent=4,
    )

print(f"Resultados almacenados en: {ida_results_path}")

Resultados almacenados en: E:\Trabajo final UGR\communication-intelligence-assistant\evaluacion_ida\caso_01\resultados_ida.json


### Hallazgos y limitaciones de la dimensión de análisis

La dimensión de análisis alcanzó un desempeño del 71,43%, superando el umbral de aceptación experimental del 70%.

El asistente seleccionó correctamente las herramientas MCP en las tres consultas experimentales y transmitió fielmente los resultados recuperados. De las siete unidades de evaluación, cinco coincidieron con los datos de referencia.

Se identificó una propagación del error predictivo del modelo de sentimientos hacia los resultados analíticos, afectando la distribución de las categorías positiva y neutral. Por otro lado, las limitaciones del modelo NER no afectaron el ranking de las tres organizaciones principales.

Estos resultados evidencian la dependencia del desempeño analítico del asistente respecto de la calidad de los resultados generados por los componentes de PLN.

Como limitación experimental, se reconoce la reducida cantidad de consultas y unidades de evaluación, lo que restringe la generalización de los resultados.

----

## Evaluación de la dimensión de síntesis (S)

Se evalúa la capacidad del asistente para generar
síntesis fundamentadas en las noticias originales,
considerando la fidelidad factual y la cobertura
informativa.

Las consultas se ejecutaron en una misma conversación,
permitiendo la reutilización del contexto acumulado.

Se conserva la ejecución inicial de S01 con búsqueda
léxica como evidencia experimental complementaria.
El cálculo principal utiliza las consultas con
búsqueda semántica explícita.

In [31]:
# Registrar los resultados de las consultas de síntesis.
synthesis_evaluation_df = pd.DataFrame(
    [
        {
            "query_id": "S01",
            "strategy": "lexical_search",
            "generated_claims": 0,
            "supported_claims": 0,
            "expected_facts": 2,
            "correct_facts": 0,
            "include_in_ida": False,
        },
        {
            "query_id": "S01",
            "strategy": "semantic_search",
            "generated_claims": 7,
            "supported_claims": 7,
            "expected_facts": 2,
            "correct_facts": 1,
            "include_in_ida": True,
        },
        {
            "query_id": "S02",
            "strategy": "semantic_search",
            "generated_claims": 22,
            "supported_claims": 22,
            "expected_facts": 3,
            "correct_facts": 1,
            "include_in_ida": True,
        },
        {
            "query_id": "S03",
            "strategy": "context_reuse",
            "generated_claims": 17,
            "supported_claims": 15,
            "expected_facts": 3,
            "correct_facts": 1,
            "include_in_ida": True,
        },
    ]
)

display(synthesis_evaluation_df)

,query_id,strategy,generated_claims,supported_claims,expected_facts,correct_facts,include_in_ida
0,S01,lexical_search,0,0,2,0,False
1,S01,semantic_search,7,7,2,1,True
2,S02,semantic_search,22,22,3,1,True
3,S03,context_reuse,17,15,3,1,True


### Cálculo de los indicadores de síntesis

Se calculan los indicadores de fidelidad factual (S1)
y cobertura informativa (S2) a partir de los resultados
de las consultas experimentales.

La dimensión de síntesis (S) se obtiene mediante la
media aritmética de ambos indicadores y se contrasta
con el umbral de aceptación establecido.

In [32]:
# Seleccionar las ejecuciones incluidas en el IDA.
synthesis_ida_df = synthesis_evaluation_df.loc[
    synthesis_evaluation_df["include_in_ida"]
]

# Consolidar las unidades de evaluación.
generated_claims = int(synthesis_ida_df["generated_claims"].sum())
supported_claims = int(synthesis_ida_df["supported_claims"].sum())
expected_facts = int(synthesis_ida_df["expected_facts"].sum())
correct_facts = int(synthesis_ida_df["correct_facts"].sum())

# Validar la consistencia de los resultados.
assert 0 <= supported_claims <= generated_claims
assert 0 <= correct_facts <= expected_facts

# Calcular los indicadores de síntesis.
s1 = supported_claims / generated_claims if generated_claims else 0.0
s2 = correct_facts / expected_facts if expected_facts else 0.0

# Calcular la dimensión de síntesis.
s = (s1 + s2) / 2

# Evaluar el cumplimiento del criterio de aceptación.
synthesis_accepted = s >= IDA_THRESHOLD

print(f"S1 - Fidelidad factual: {s1:.4f}")
print(f"S2 - Cobertura informativa: {s2:.4f}")
print(f"S  - Síntesis: {s:.4f}")
print(f"Criterio de aceptación: {synthesis_accepted}")

S1 - Fidelidad factual: 0.9565
S2 - Cobertura informativa: 0.3750
S  - Síntesis: 0.6658
Criterio de aceptación: False


In [33]:
# Recuperar los resultados anteriores del IDA.
ida_results = load_ground_truth(ida_results_path)

# Incorporar los resultados de la dimensión de síntesis.
ida_results["synthesis"] = {
    "s1_fidelity": s1,
    "s2_coverage": s2,
    "score": s,
    "threshold": IDA_THRESHOLD,
    "accepted": bool(synthesis_accepted),
    "generated_claims": generated_claims,
    "supported_claims": supported_claims,
    "expected_facts": expected_facts,
    "correct_facts": correct_facts,
    "queries": synthesis_evaluation_df.to_dict(orient="records"),
    "evaluation_mode": "conversational",
}

# Persistir los resultados experimentales.
with ida_results_path.open("w", encoding="utf-8") as file:
    json.dump(
        ida_results,
        file,
        ensure_ascii=False,
        indent=4,
    )

print(f"Resultados almacenados en: {ida_results_path}")

Resultados almacenados en: E:\Trabajo final UGR\communication-intelligence-assistant\evaluacion_ida\caso_01\resultados_ida.json


### Hallazgos y limitaciones de la dimensión de síntesis

La dimensión de síntesis alcanzó un desempeño del 66,58%,
inferior al umbral de aceptación experimental del 70%.

El asistente obtuvo una fidelidad factual del 95,65%,
respaldando 44 de las 46 afirmaciones generadas.
Sin embargo, la cobertura informativa alcanzó el 37,50%,
al recuperar completamente solo tres de los ocho hechos
clave esperados.

Se identificaron las siguientes limitaciones:

- La búsqueda léxica inicial no recuperó la noticia
  solicitada, pese a encontrarse almacenada.
- La búsqueda semántica explícita permitió recuperar
  correctamente las noticias objetivo.
- Las síntesis omitieron aspectos cualitativos presentes
  en los documentos originales.
- Se identificaron dos afirmaciones sin respaldo factual.

Durante la evaluación se modificaron los prompts para
orientar al asistente hacia la búsqueda semántica,
conservando la ejecución inicial como evidencia
complementaria.

Las consultas se realizaron dentro de una misma
conversación, permitiendo la reutilización del contexto,
lo que limita la independencia experimental.

Asimismo, la evaluación manual introduce un componente
de subjetividad en la identificación de afirmaciones
y la valoración de su correspondencia semántica.

Los resultados indican que el asistente presenta una
elevada fidelidad factual, pero requiere mejoras en
la cobertura de información relevante.

## Evaluación integral del asistente (IDA)

Se calcula el Índice de Desempeño del Asistente (IDA)
mediante la media aritmética de las dimensiones de
monitoreo, análisis y síntesis, asignando la misma
ponderación a cada una.

El criterio de aceptación requiere que tanto el índice
global como cada dimensión alcancen el umbral
experimental establecido.

In [34]:
# Recuperar los resultados de las dimensiones evaluadas.
ida_results = load_ground_truth(ida_results_path)

m = ida_results["monitoring"]["score"]
a = ida_results["analysis"]["score"]
s = ida_results["synthesis"]["score"]

# Validar el rango de las dimensiones.
assert all(0 <= score <= 1 for score in (m, a, s))

# Calcular el índice de desempeño integral.
ida = (m + a + s) / 3

# Evaluar los criterios de aceptación.
ida_accepted = ida >= IDA_THRESHOLD
dimensions_accepted = all(
    score >= IDA_THRESHOLD for score in (m, a, s)
)

case_accepted = ida_accepted and dimensions_accepted

print(f"M   - Monitoreo: {m:.4f}")
print(f"A   - Análisis: {a:.4f}")
print(f"S   - Síntesis: {s:.4f}")
print(f"IDA - Desempeño integral: {ida:.4f}")
print(f"Umbral global alcanzado: {ida_accepted}")
print(f"Dimensiones aceptadas: {dimensions_accepted}")
print(f"Criterio de aceptación del caso: {case_accepted}")

M   - Monitoreo: 0.8095
A   - Análisis: 0.7143
S   - Síntesis: 0.6658
IDA - Desempeño integral: 0.7299
Umbral global alcanzado: True
Dimensiones aceptadas: False
Criterio de aceptación del caso: False


In [35]:
# Incorporar el resultado integral del asistente.
ida_results["ida"] = {
    "score": ida,
    "threshold": IDA_THRESHOLD,
    "global_accepted": bool(ida_accepted),
    "dimensions_accepted": bool(dimensions_accepted),
    "case_accepted": bool(case_accepted),
}

# Persistir los resultados consolidados.
with ida_results_path.open("w", encoding="utf-8") as file:
    json.dump(
        ida_results,
        file,
        ensure_ascii=False,
        indent=4,
    )

print(f"Resultados almacenados en: {ida_results_path}")

Resultados almacenados en: E:\Trabajo final UGR\communication-intelligence-assistant\evaluacion_ida\caso_01\resultados_ida.json
